# Модуль 1. Разбиение данных и кросс-валидация

**Длительность:** 90 минут  
**Формат:** теория (60 мин) + практика (30 мин)  
**Цель модуля:** Учащийся должен понять, почему для оценки модели нужны отдельные выборки, освоить механику K-Fold и StratifiedKFold, и научиться гарантировать сохранение пропорций классов при разбиении несбалансированных данных.

## 1. Постановка задачи: зачем разбивать данные (10 мин)

### 1.1. Проблема оценки на обучающих данных
В Модуле 0 установлено: высокая точность на обучающей выборке не гарантирует, что модель выявила закономерность. Чтобы проверить обобщающую способность, нужно оценить качество модели на объектах, которые она не видела в процессе обучения.

Для этого исходный набор данных разбивается на части. Модель обучается на одной части, а оценивается на другой.

### 1.2. Требования к процедуре оценки

Процедура оценки должна удовлетворять двум условиям:

1. **Объекты для оценки не должны участвовать в обучении.** Иначе оценка будет завышена, поскольку модель уже «видела» правильные ответы для этих объектов.
2. **Оценка должна быть устойчивой.** Одно случайное разбиение может дать аномально высокую или низкую метрику из-за случайного состава подвыборок. Нужна процедура, которая усредняет эффект случайности.

## 2. Train / Validation / Test: зачем три выборки (15 мин)

### 2.1. Определения

- **Обучающая выборка (Train):** данные, на которых модель настраивает свои параметры (веса, структуру дерева и т.д.).
- **Валидационная выборка (Validation):** данные, на которых проверяется качество модели в процессе разработки. Используется для сравнения разных моделей, подбора гиперпараметров, ранней остановки обучения.
- **Тестовая выборка (Test):** данные, которые модель не видит ни на одном этапе разработки. Используются один раз в самом конце для финальной оценки.

### 2.2. Почему недостаточно двух выборок
Если существует только Train и Test, и мы используем Test для выбора между моделями или настройки гиперпараметров, то информация из Test неявно просачивается в процесс принятия решений.

Пример: обучено 10 моделей с разными гиперпараметрами. Мы выбираем ту, у которой метрика на Test выше. Фактически, мы подстроили выбор модели под конкретный набор объектов в Test. При этом Test перестал быть независимым.

Результат: метрика на Test перестает отражать реальное качество на новых данных. Она завышена.

### 2.3. Роль трех выборок
- **Train** — для обучения параметров.
- **Validation** — для принятия инженерных решений (какая модель лучше, какие гиперпараметры выбрать).
- **Test** — для единственной финальной проверки, после которой модель не дорабатывается.

Если метрика на Test неудовлетворительна, правильный путь — вернуться к разработке, но при этом Test уже скомпрометирован. Нужен новый Test. На практике это означает, что Test выделяется один раз и используется строго в конце.

## 3. Hold-out: простое разбиение (10 мин)

### 3.1. Механика
Самый простой способ: случайное разбиение исходного набора на две или три части в заданной пропорции.

Стандартные пропорции:
- Train / Test: 80 % / 20 %
- Train / Validation / Test: 70 % / 15 % / 15 % или 60 % / 20 % / 20 %

Реализация в Scikit-Learn:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 3.2. Проблема hold-out
Оценка зависит от конкретного случайного разбиения. Если данные малы или имеют выбросы, одно разбиение может дать слишком оптимистичную или пессимистичную метрику.

Пример: в выборке 1000 объектов, из них 50 — редкий класс. При случайном разбиении 80/20 в тест может попасть 4 редких объекта или 14. Разница в 10 объектов существенно изменит метрику.

Hold-out дает **одну** оценку. Мы не знаем, насколько она устойчива.

## 4. K-Fold кросс-валидация (15 мин)

### 4.1. Идея
Вместо одного разбиения выполнить K разбиений, обучить K моделей, получить K метрик и усреднить их. Это снижает влияние случайного состава одной конкретной тестовой выборки.

### 4.2. Механика пошагово

Пусть K = 5.

1. Вся выборка делится на 5 равных (по размеру) блоков — фолдов.
2. Итерация 1: фолд 1 — тест, фолды 2–5 — обучение. Обучаем модель, считаем метрику на фолде 1.
3. Итерация 2: фолд 2 — тест, фолды 1, 3–5 — обучение. Считаем метрику на фолде 2.
4. Повторяем для всех 5 фолдов.
5. Итоговая оценка — среднее 5 метрик.

Каждый объект ровно один раз побывал в тестовой части и K-1 раз в обучающей.

### 4.3. Почему это лучше hold-out
- Усреднение по K итерациям снижает дисперсию оценки.
- Каждый объект участвует и в обучении, и в валидации. Нет потери данных для обучения (в отличие от hold-out, где 20 % выборки навсегда отданы под тест).
- Особенно важно при малых выборках.

### 4.4. Выбор K
- K = 5 или K = 10 — стандарт на практике. Баланс между стоимостью вычислений (K обучений) и устойчивостью оценки.
- K = n (leave-one-out): каждый объект — отдельный фолд. Очень дорого, но иногда используется при крайне малых выборках.

### 4.5. Код

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f"Scores: {scores}")
print(f"Mean: {scores.mean():.3f}, Std: {scores.std():.3f}")

## 5. StratifiedKFold: сохранение пропорций классов (20 мин)

### 5.1. Проблема обычного K-Fold при дисбалансе
K-Fold делит выборку на равные по размеру фолды случайным образом. Если классы сильно несбалансированы (например, 1 % positive, 99 % negative), случайное разбиение может привести к тому, что в некоторый фолд не попадет ни одного positive объекта.

Если в валидационном фолде нет positive объектов, метрики Precision, Recall, F1 не определены (деление на ноль), а Accuracy искажена. Обучающий фолд, в свою очередь, может содержать все positive объекты, кроме одного.

### 5.2. Решение: стратификация
**Стратифицированное разбиение (StratifiedKFold)** гарантирует, что в каждом фолде пропорция классов повторяет пропорцию классов в исходной выборке.

Если в полной выборке 1 % positive, то и в каждом обучающем фолде, и в каждом тестовом фолде будет ровно 1 % positive.

### 5.3. Механика

Алгоритм работает отдельно для каждого класса:
1. Объекты каждого класса перемешиваются.
2. Каждый класс делится на K равных частей.
3. Фолд i формируется взятием i-й части от каждого класса.

Таким образом, пропорции сохраняются в каждом фолде.

### 5.4. Почему это критично
- **Валидация:** если тестовый фолд не содержит positive объектов, мы не можем оценить, как модель справляется с редким классом. Это именно тот класс, который обычно представляет бизнес-интерес (фрод, болезнь, отток).
- **Обучение:** если обучающий фолд содержит слишком мало positive объектов, модель не получает достаточно информации для выявления закономерности.

### 5.5. Код

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print(f"Mean accuracy: {scores.mean():.3f}")

## 6. Практическая часть: кастомный валидатор (30 мин)

### 6.1. Постановка задачи

Написать класс `CustomStratifiedValidator` в строгом ООП-стиле, который:
- Принимает на вход данные с сильным дисбалансом классов (1 % positive).
- Гарантирует строго одинаковое распределение классов в каждом обучающем и тестовом фолде.
- Проверяет корректность: если в каком-либо фолде пропорция классов отличается от исходной более чем на допустимый epsilon, выбрасывает исключение.

### 6.2. Код решения

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from collections import Counter


class CustomStratifiedValidator:
    """
    Кастомный валидатор на основе StratifiedKFold.
    Гарантирует сохранение пропорций классов в каждом фолде.
    """
    
    def __init__(self, n_splits=5, shuffle=True, random_state=None, tol=1e-6):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
        self.tol = tol  # допустимое отклонение пропорции
    
    def split(self, X, y):
        """
        Генератор, возвращающий (train_indices, test_indices) для каждого фолда.
        """
        y = np.array(y)
        classes, counts = np.unique(y, return_counts=True)
        n_samples = len(y)
        
        # Исходные пропорции классов
        original_proportions = {cls: cnt / n_samples for cls, cnt in zip(classes, counts)}
        
        # Индексы объектов по классам
        class_indices = {cls: np.where(y == cls)[0] for cls in classes}
        
        # Перемешивание внутри каждого класса
        rng = np.random.RandomState(self.random_state)
        for cls in classes:
            rng.shuffle(class_indices[cls])
        
        # Разбиение каждого класса на n_splits частей
        fold_indices_by_class = {}
        for cls in classes:
            indices = class_indices[cls]
            fold_sizes = np.full(self.n_splits, len(indices) // self.n_splits)
            fold_sizes[:len(indices) % self.n_splits] += 1  # распределяем остаток
            
            current = 0
            folds = []
            for size in fold_sizes:
                folds.append(indices[current:current + size])
                current += size
            fold_indices_by_class[cls] = folds
        
        # Сборка фолдов
        for fold_idx in range(self.n_splits):
            test_indices = []
            train_indices = []
            
            for cls in classes:
                # Тестовая часть — i-й фолд класса
                test_part = fold_indices_by_class[cls][fold_idx]
                # Обучающая часть — все остальные фолды класса
                train_parts = [
                    fold_indices_by_class[cls][j] 
                    for j in range(self.n_splits) if j != fold_idx
                ]
                
                test_indices.extend(test_part)
                for part in train_parts:
                    train_indices.extend(part)
            
            test_indices = np.array(test_indices)
            train_indices = np.array(train_indices)
            
            # Проверка пропорций
            self._validate_proportions(y, train_indices, test_indices, original_proportions)
            
            yield train_indices, test_indices
    
    def _validate_proportions(self, y, train_idx, test_idx, original_props):
        """
        Проверяет, что пропорции классов в train и test совпадают с исходными.
        """
        for idx_set, name in [(train_idx, 'train'), (test_idx, 'test')]:
            classes, counts = np.unique(y[idx_set], return_counts=True)
            proportions = dict(zip(classes, counts / len(idx_set)))
            
            for cls, orig_prop in original_props.items():
                actual_prop = proportions.get(cls, 0.0)
                if abs(actual_prop - orig_prop) > self.tol:
                    raise ValueError(
                        f"Нарушение стратификации в {name}: "
                        f"класс {cls}, ожидалось {orig_prop:.6f}, получено {actual_prop:.6f}"
                    )


# === Демонстрация ===

# 1. Генерация несбалансированного датасета: 1% positive
X, y = make_classification(
    n_samples=50000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.99, 0.01],  # 99% класс 0, 1% класс 1
    flip_y=0,
    random_state=42
)

print(f"Распределение классов в полной выборке: {Counter(y)}")
print(f"Пропорция класса 1: {np.mean(y == 1):.4f}")

# 2. Использование кастомного валидатора
validator = CustomStratifiedValidator(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, test_idx) in enumerate(validator.split(X, y)):
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_prop = np.mean(y_train == 1)
    test_prop = np.mean(y_test == 1)
    
    print(f"\nФолд {fold_idx + 1}:")
    print(f"  Train: {len(y_train)} объектов, класс 1: {train_prop:.4f}")
    print(f"  Test:  {len(y_test)} объектов, класс 1: {test_prop:.4f}")

### 6.3. Что обсудить по результатам
- В каждом фолде пропорция класса 1 ровно 0.0100 (1 %), как и в исходной выборке.
- Если бы мы использовали обычный `KFold`, пропорции флуктуировали бы.
- Класс `CustomStratifiedValidator` воспроизводит логику `StratifiedKFold` из Scikit-Learn, но делает её явной и проверяемой.

## 7. Итоги модуля (10 мин)

### Ключевые тезисы
1. Для оценки модели нужны данные, которые она не видела при обучении.
2. Три выборки: Train (обучение), Validation (выбор модели и гиперпараметров), Test (финальная проверка). Test нельзя использовать для принятия решений в процессе разработки.
3. Hold-out прост, но дает неустойчивую оценку, зависящую от случайного разбиения.
4. K-Fold усредняет оценку по K итерациям, повышая надёжность.
5. При несбалансированных классах обычный K-Fold опасен: редкий класс может выпасть из фолда. StratifiedKFold гарантирует сохранение пропорций классов в каждом фолде.
6. Стратификация — это не «улучшение», а **требование** при дисбалансе. Без неё оценка качества на редком классе невозможна.

### Контрольные вопросы
- Почему нельзя использовать тестовую выборку для выбора гиперпараметров?
- В чём преимущество K-Fold перед hold-out?
- Почему StratifiedKFold важен при дисбалансе классов? Что произойдёт, если его не применить?
- Как алгоритмически обеспечить сохранение пропорций классов при разбиении?

### Что дальше
В следующем модуле мы изучим метрики классификации. Мы увидим, что даже при правильном разбиении выбор метрики может сделать хорошую модель бесполезной — особенно при дисбалансе классов.